In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
train = pd.read_csv("./data/mpg_train.csv")
test = pd.read_csv("./data/mpg_test.csv") # no mpg column -> we must predict it
train

,id,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,1,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,3,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
2,4,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
3,6,15.0,8,429.0,198.0,4341,10.0,70,usa,ford galaxie 500
4,7,14.0,8,454.0,220.0,4354,9.0,70,usa,chevrolet impala
...,...,...,...,...,...,...,...,...,...,...
313,394,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
314,395,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
315,396,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
316,397,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger


In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            318 non-null    int64  
 1   mpg           318 non-null    float64
 2   cylinders     318 non-null    int64  
 3   displacement  318 non-null    float64
 4   horsepower    315 non-null    float64
 5   weight        318 non-null    int64  
 6   acceleration  318 non-null    float64
 7   model_year    318 non-null    int64  
 8   origin        318 non-null    str    
 9   name          318 non-null    str    
dtypes: float64(4), int64(4), str(2)
memory usage: 25.0 KB


In [4]:
train.isna().sum()

id              0
mpg             0
cylinders       0
displacement    0
horsepower      3
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

In [5]:
test.isna().sum() # horsepower missing here too -> must fill, can't drop

id              0
cylinders       0
displacement    0
horsepower      3
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

In [6]:
hp_median = train["horsepower"].median() # learn from train only

train["horsepower"] = train["horsepower"].fillna(hp_median)
test["horsepower"] = test["horsepower"].fillna(hp_median) # same value for test

In [7]:
fix_brand = {"chevroelt": "chevrolet", "chevy": "chevrolet", "vw": "volkswagen", "vokswagen": "volkswagen",
	"toyouta": "toyota", "maxda": "mazda", "mercedes-benz": "mercedes"}

train["brand"] = train["name"].str.split().str[0].replace(fix_brand)
test["brand"] = test["name"].str.split().str[0].replace(fix_brand)

In [8]:
le_origin, le_brand = LabelEncoder(), LabelEncoder()

train["origin"] = le_origin.fit_transform(train["origin"]) # europe, japan, usa -> 0, 1, 2
test["origin"] = le_origin.transform(test["origin"]) # transform only, same mapping

train["brand"] = le_brand.fit_transform(train["brand"])
test["brand"] = le_brand.transform(test["brand"])

In [9]:
X = train.drop(columns=["id", "name", "mpg"])
y = train[["mpg"]] # double brackets -> shape (n, 1), matches model output
X_sub = test.drop(columns=["id", "name"]) # features of the data to submit

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
	X, y, test_size=0.2, random_state=42)

In [11]:
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train) # fit on TRAIN only
Xval = scaler.transform(X_val)
Xsub = scaler.transform(X_sub)

In [12]:
Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
Xval_t = torch.tensor(Xval, dtype=torch.float32)
Xsub_t = torch.tensor(Xsub, dtype=torch.float32)
ytr_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32) # (n, 1)

train_ds = TensorDataset(Xtr_t, ytr_t) # pairs (x, y) row by row
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True) # gives mini-batches of 32 rows

In [13]:
xb, yb = next(iter(train_loader)) # look at one batch
xb.shape, yb.shape

(torch.Size([32, 8]), torch.Size([32, 1]))

In [14]:
class Model(nn.Module):
	def __init__(self, n_in, n_hidden, n_out):
		super().__init__()
		self.fc1 = nn.Linear(n_in, n_hidden)
		self.fc2 = nn.Linear(n_hidden, n_out)

	def forward(self, x):
		return self.fc2(torch.relu(self.fc1(x)))

In [20]:
Xtr.shape

(254, 8)

In [15]:
torch.manual_seed(42)
model = Model(n_in=Xtr.shape[1], n_hidden=20, n_out=1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 300
history = []

for epoch in range(epochs):
	model.train()
	total = 0
	for xb, yb in train_loader: # one mini-batch at a time
		y_hat = model(xb) # 1. forward
		loss = loss_fn(y_hat, yb) # 2. loss
		optimizer.zero_grad() # clear old gradients
		loss.backward() # 3. backward
		optimizer.step() # 4. update weights
		total += loss.item() * len(xb)
	history.append(total / len(train_ds)) # average loss of the epoch
	if epoch % 50 == 0:
		print(f"epoch {epoch:3d}  loss {history[-1]:.4f}")

epoch   0  loss 605.2513
epoch  50  loss 8.1321
epoch 100  loss 7.5540
epoch 150  loss 7.0055
epoch 200  loss 6.5541
epoch 250  loss 6.4144


In [16]:
model.eval()
with torch.no_grad(): # no gradients needed for predicting
	y_pred_train = model(Xtr_t).numpy()
	y_pred_val = model(Xval_t).numpy()

print(f"Train  RMSE: {root_mean_squared_error(y_train, y_pred_train):.2f}  R2: {r2_score(y_train, y_pred_train):.3f}")
print(f"Val    RMSE: {root_mean_squared_error(y_val, y_pred_val):.2f}  R2: {r2_score(y_val, y_pred_val):.3f}")

Train  RMSE: 2.47  R2: 0.904
Val    RMSE: 2.09  R2: 0.924


In [17]:
SAVE_CSV = False

submission_df = test[["id"]].copy()
with torch.no_grad():
	submission_df["mpg"] = model(Xsub_t).numpy().ravel() # (n, 1) -> (n,)
if SAVE_CSV:
	submission_df.to_csv("submission.csv", index=False)

submission_df

,id,mpg
0,2,14.730414
1,5,17.487617
2,13,16.070910
3,16,21.304804
4,18,21.272337
...,...,...
75,379,37.010498
76,380,34.901123
77,387,26.743824
78,390,25.636637
